# Agent Lab â€” Helpdesk IT Parc

Ce notebook compare deux faÃ§ons d'utiliser le mÃªme agent Google ADK :

1. **Sans mÃ©moire** : une nouvelle session Ã  chaque question.
2. **Avec mÃ©moire** : une seule session rÃ©utilisÃ©e.

Toutes les donnÃ©es sont fictives.

## PrÃ©paration
On charge la clÃ© depuis `.env` avec `python-dotenv`. La clÃ© n'est jamais affichÃ©e : on vÃ©rifie seulement sa prÃ©sence.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# ADK / google-genai lisent GOOGLE_API_KEY ; on accepte aussi GEMINI_API_KEY.
if not os.getenv("GOOGLE_API_KEY") and os.getenv("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
print("ClÃ© prÃ©sente :", bool(os.getenv("GOOGLE_API_KEY")))

## L'agent Â« helpdesk IT Parc Â»
L'agent reÃ§oit une courte base de connaissances fictive directement dans ses `instructions`. Les dÃ©lais sont des espaces rÃ©servÃ©s `[dÃ©lai Ã  dÃ©finir]`.

In [ ]:
from google.adk.agents.llm_agent import Agent

MODEL = "gemini-3.6-flash"  # modèle Flash disponible ; changer ici si besoin

INSTRUCTIONS = """
Tu es l'assistant du helpdesk IT Parc d'une organisation fictive.
Tu aides le personnel Ã  choisir la bonne demande informatique. RÃ©ponds en franÃ§ais, de faÃ§on brÃ¨ve.
Appuie-toi uniquement sur la base de connaissances ci-dessous. Si une information manque, dis-le.

BASE DE CONNAISSANCES (fictive)
- Nouvel Ã©quipement : demande d'un poste, Ã©cran ou accessoire pour un nouvel arrivant ou un besoin nouveau.
  Contact : Ã©quipe Approvisionnement IT (approvisionnement-it@exemple.test). DÃ©lai typique : [dÃ©lai Ã  dÃ©finir].
- Panne : un Ã©quipement qui ne fonctionne plus (ne s'allume pas, Ã©cran cassÃ©, etc.).
  Contact : support de proximitÃ© (support-parc@exemple.test). DÃ©lai typique : [dÃ©lai Ã  dÃ©finir].
- RÃ©affectation : transfert d'un Ã©quipement existant d'une personne ou d'un service Ã  un autre.
  Contact : gestion du parc (gestion-parc@exemple.test). DÃ©lai typique : [dÃ©lai Ã  dÃ©finir].
- Retour : restitution d'un Ã©quipement devenu inutile ou en fin de contrat.
  Contact : gestion du parc (gestion-parc@exemple.test). DÃ©lai typique : [dÃ©lai Ã  dÃ©finir].
"""

helpdesk_agent = Agent(
    model=MODEL,
    name="helpdesk_it_parc",
    description="Oriente le personnel vers la bonne demande IT.",
    instruction=INSTRUCTIONS,
)

## Utilitaire : poser une question
Un **Runner** exÃ©cute l'agent. Il a besoin d'un **service de sessions** : une *session* est la conversation en cours (identifiant, historique des messages et Ã©tat). L'agent ne Â« voit Â» que l'historique de la session qu'on lui donne.

La fonction ci-dessous est asynchrone : elle envoie la question dans une session donnÃ©e et renvoie la rÃ©ponse finale.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "agent_lab"
USER_ID = "utilisateur_demo"

import asyncio
from google.genai.errors import ServerError

async def ask(runner, session_id, question, retries=5):
    message = types.Content(role="user", parts=[types.Part(text=question)])
    answer = ""
    for attempt in range(retries):
        try:
            async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=message):
                if event.is_final_response() and event.content and event.content.parts:
                    answer = "".join(p.text or "" for p in event.content.parts)
            break
        except ServerError:  # 503 « forte demande » : on réessaie après une pause
            if attempt == retries - 1:
                raise
            await asyncio.sleep(5 * (attempt + 1))
    print(f"Q : {question}\nR : {answer}\n")
    return answer

# Section 1 â€” Sans mÃ©moire
**Pourquoi l'agent oublie-t-il ?** Ici on crÃ©e une **nouvelle session Ã  chaque question**. Chaque session dÃ©marre vide : la deuxiÃ¨me question arrive sans aucun historique, donc Â« Ã§a Â» n'a pas de rÃ©fÃ©rent.

In [ ]:
session_service_1 = InMemorySessionService()
runner_1 = Runner(agent=helpdesk_agent, app_name=APP_NAME, session_service=session_service_1)

questions = [
    "Mon PC portable ne s'allume plus depuis ce matin.",
    "Quel type de demande je dois ouvrir pour Ã§a ?",
]

for q in questions:
    session = await session_service_1.create_session(app_name=APP_NAME, user_id=USER_ID)  # session neuve
    await ask(runner_1, session.id, q)

**Constat :** Ã  la deuxiÃ¨me question, l'agent ne sait pas Ã  quoi Â« Ã§a Â» renvoie : il demande des prÃ©cisions ou rÃ©pond de faÃ§on gÃ©nÃ©rale, car la premiÃ¨re question n'est pas dans sa session.

# Section 2 â€” Avec mÃ©moire
**Ce qui change :** on crÃ©e **une seule session** et on la **rÃ©utilise** pour les deux questions. Le `InMemorySessionService` conserve l'historique en mÃ©moire (il disparaÃ®t Ã  l'arrÃªt du kernel) et le Runner le transmet Ã  l'agent Ã  chaque tour.

In [ ]:
session_service_2 = InMemorySessionService()
runner_2 = Runner(agent=helpdesk_agent, app_name=APP_NAME, session_service=session_service_2)

shared_session = await session_service_2.create_session(app_name=APP_NAME, user_id=USER_ID)  # une seule session

for q in questions:
    await ask(runner_2, shared_session.id, q)

**Constat :** l'agent se souvient de la panne du PC portable et rÃ©pond que la demande Ã  ouvrir est une **panne**.

*Rappel : `InMemorySessionService` sert au dÃ©veloppement ; en production on utiliserait un service de sessions persistant.*